# Alter `telemetry.agent_telemetry`

Scratchpad for one-off, additive schema changes to the live table --
adding a new column, not modifying or renaming an existing one (that
still needs `notebooks/delete_agent_telemetry_table.ipynb` + `create_table.py`,
since BigQuery can't `ALTER` a column's mode or rename it).

In [3]:
import sys
sys.path.insert(0, "..")

from google.cloud import bigquery

from app.config import GCP_PROJECT_ID, TELEMETRY_DATASET, TELEMETRY_TABLE

client = bigquery.Client(project=GCP_PROJECT_ID)
table_ref = f"{GCP_PROJECT_ID}.{TELEMETRY_DATASET}.{TELEMETRY_TABLE}"

## Add `all_prose_numeric_claims` (Phase 3 item 6)

The model's declared prose numeric claims, from `submit_answer` -- a new
`REPEATED FLOAT` column, matching `app/telemetry/schema.py`.

In [2]:
table = client.get_table(table_ref)
table.schema = table.schema + [bigquery.SchemaField("all_prose_numeric_claims", "FLOAT", mode="REPEATED")]
table = client.update_table(table, ["schema"])
print(f"{len(table.schema)} top-level fields now")

BadRequest: 400 PATCH https://bigquery.googleapis.com/bigquery/v2/projects/instacart-ml-model/datasets/telemetry/tables/agent_telemetry?prettyPrint=false: Field all_prose_numeric_claims already exists in schema

## Add `clarifying_question` (clarifying-question feature)

The clarifying question the agent asked instead of answering, from
`submit_answer` -- a new nullable `STRING` column, matching
`app/telemetry/schema.py`. NULL on every answer turn.

In [2]:
table = client.get_table(table_ref)
table.schema = table.schema + [bigquery.SchemaField("clarifying_question", "STRING", mode="NULLABLE")]
table = client.update_table(table, ["schema"])
print(f"{len(table.schema)} top-level fields now")

31 top-level fields now


In [4]:
print(table.schema)

[SchemaField('conversation_id', 'STRING', 'REQUIRED', None, None, (), None, None), SchemaField('user_id', 'STRING', 'REQUIRED', None, None, (), None, None), SchemaField('question', 'STRING', 'REQUIRED', None, None, (), None, None), SchemaField('answer_markdown', 'STRING', 'REQUIRED', None, None, (), None, None), SchemaField('turn_started_at', 'TIMESTAMP', 'REQUIRED', None, None, (), None, None), SchemaField('turn_completed_at', 'TIMESTAMP', 'REQUIRED', None, None, (), None, None), SchemaField('filter_context', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('active_page', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('pending_query', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('estimated_cost', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('approval_decision', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('verified', 'BOOLEAN', 'REQUIRED', None, None, (), None, None), SchemaField('verification_retry_co

## Add billed bytes to tool calls and pending queries

`tool_calls.bytes_billed` (billed bytes per BigQuery call) and
`pending_approval`-style `pending_queries.estimated_bytes` (dry-run estimate).
Both are nullable nested fields added to existing RECORDs.

In [4]:
table = client.get_table(table_ref)


def add_subfield(schema, record_name, field):
    out = []
    for f in schema:
        if f.name == record_name:
            f = bigquery.SchemaField(f.name, f.field_type, mode=f.mode, fields=list(f.fields) + [field])
        out.append(f)
    return out


table.schema = add_subfield(table.schema, "tool_calls", bigquery.SchemaField("bytes_billed", "INTEGER", mode="NULLABLE"))
table.schema = add_subfield(table.schema, "pending_queries", bigquery.SchemaField("estimated_bytes", "INTEGER", mode="NULLABLE"))
table = client.update_table(table, ["schema"])
print("updated")

updated
